# Trabajo Práctico — Fundamentos de la IA
## Clasificación de dirección de precios con Regresión Logística vs. Perceptrón Multicapa (MLP)

**Maestría en Inteligencia Artificial — Universidad de Palermo**

---

### Resumen

Este trabajo demuestra fundamentos de aprendizaje automático aplicados a un problema
financiero: **predecir la dirección futura del precio de una acción** (sube / lateral / baja)
a partir de una ventana de datos históricos y de indicadores de análisis técnico.

El eje del trabajo es una **comparativa metodológica** entre dos modelos de complejidad
creciente, siguiendo el arco conceptual de los primeros capítulos de *Deep Learning*
(Goodfellow, Bengio & Courville, 2016):

1. **Regresión Logística** — modelo lineal de referencia (Cap. 5, *Machine Learning Basics*).
2. **Perceptrón Multicapa (MLP)** — red neuronal feedforward con no-linealidad (Cap. 6, *Deep Feedforward Networks*).

Sobre estos modelos se ilustran además: **regularización** (Cap. 7), **optimización del
entrenamiento** (Cap. 8) y los **fundamentos probabilísticos** de la función de pérdida (Cap. 3).

> **Aclaración metodológica importante:** este trabajo NO pretende construir un sistema
> rentable de trading. La predicción de dirección de precios es un problema notoriamente
> difícil y cercano al azar en mercados eficientes. El objetivo es **pedagógico**: mostrar
> cómo se plantea, entrena, regulariza y evalúa correctamente un clasificador, y cómo se
> comparan modelos de forma honesta evitando *data leakage*.


## 1. Marco conceptual

### 1.1 El problema de aprendizaje (Goodfellow, Cap. 5)

Goodfellow define una tarea de aprendizaje automático mediante tres componentes: la **tarea T**,
la **medida de desempeño P** y la **experiencia E**.

| Componente | En este trabajo |
|---|---|
| **Tarea T** | Clasificar la dirección del precio a *k* días vista en 3 clases: `SUBE`, `LATERAL`, `BAJA`. |
| **Desempeño P** | *Accuracy* y *F1-score* macro sobre un conjunto de prueba fuera de muestra. |
| **Experiencia E** | Ventanas históricas de precios (OHLCV) + indicadores técnicos de una acción del mercado. |

### 1.2 ¿Por qué comparar un modelo lineal contra uno no lineal?

El **teorema de aproximación universal** (Goodfellow, Cap. 6.4.1) establece que una red
feedforward con al menos una capa oculta y suficientes neuronas puede aproximar cualquier
función continua. La regresión logística, en cambio, sólo puede trazar **fronteras de decisión
lineales**. La comparativa nos permite responder empíricamente: *¿el problema de dirección de
precios tiene estructura no lineal que un MLP pueda capturar y un modelo lineal no?*

### 1.3 Fundamento probabilístico de la pérdida (Cap. 3)

Ambos modelos terminan en una capa **softmax** que produce una distribución de probabilidad
sobre las 3 clases. El entrenamiento minimiza la **entropía cruzada** (*cross-entropy*), que
equivale a la **estimación de máxima verosimilitud** de los parámetros dado el conjunto de datos.


## 2. Configuración del entorno

Instalamos las dependencias. En Google Colab, `yfinance` y `ta` no vienen preinstalados.


In [ ]:
!pip install yfinance ta --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import ta

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

## 3. Parámetros del experimento

Todos los hiperparámetros del problema están centralizados acá para facilitar la
experimentación. Cambiar el `TICKER`, el `HORIZONTE` o el `UMBRAL` permite estudiar
cómo varía la dificultad del problema.


In [ ]:
# --- Datos ---
TICKER      = "GGAL"      # Grupo Galicia (ADR en NYSE, buena liquidez e historia)
FECHA_INI   = "2010-01-01"
FECHA_FIN   = "2024-12-31"

# --- Definición de la etiqueta ---
VENTANA     = 30          # días de historia que ve el modelo (features)
HORIZONTE   = 5           # a cuántos días predecimos la dirección (k)
UMBRAL      = 0.02        # +/- 2%: zona muerta para la clase LATERAL (theta)

# --- Split y entrenamiento ---
TEST_FRAC   = 0.20        # último 20% temporal para test (NO aleatorio)
EPOCHS      = 100
BATCH_SIZE  = 64

CLASES = {0: "BAJA", 1: "LATERAL", 2: "SUBE"}

## 4. Obtención y exploración de los datos

Descargamos el histórico OHLCV (*Open, High, Low, Close, Volume*) del ticker elegido.


In [ ]:
df = yf.download(TICKER, start=FECHA_INI, end=FECHA_FIN, auto_adjust=True)

# yfinance a veces devuelve columnas MultiIndex; las aplanamos
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df.dropna()
print(f"Filas descargadas: {len(df)}")
print(f"Rango: {df.index.min().date()} a {df.index.max().date()}")
df.head()

In [ ]:
# Visualización del precio de cierre
plt.figure(figsize=(13, 4))
plt.plot(df.index, df["Close"], linewidth=0.8)
plt.title(f"{TICKER} — Precio de cierre ajustado")
plt.ylabel("Precio (USD)")
plt.grid(alpha=0.3)
plt.show()

## 5. Ingeniería de características: indicadores de análisis técnico

En lugar de darle al modelo el precio crudo (que es no estacionario y con tendencia),
construimos **indicadores de análisis técnico** — las mismas herramientas que usa un analista
humano. Esto es *feature engineering* clásico (Goodfellow, Cap. 5.4): representar el dato de
forma que la estructura relevante sea más accesible para el modelo.

Indicadores elegidos:

- **Retornos logarítmicos** — captura el cambio relativo, es aproximadamente estacionario.
- **RSI (Relative Strength Index)** — momento; detecta sobrecompra/sobreventa.
- **MACD** — cruce de medias móviles exponenciales; detecta cambios de tendencia.
- **Bandas de Bollinger (%B)** — posición del precio respecto a su volatilidad reciente.
- **Media móvil relativa** — precio respecto a su media de 20 días.
- **Volatilidad** — desvío estándar móvil de los retornos.


In [ ]:
d = df.copy()

# Retorno logarítmico diario
d["log_ret"] = np.log(d["Close"] / d["Close"].shift(1))

# RSI (14)
d["rsi"] = ta.momentum.RSIIndicator(d["Close"], window=14).rsi()

# MACD (histograma)
macd = ta.trend.MACD(d["Close"])
d["macd_diff"] = macd.macd_diff()

# Bandas de Bollinger — %B
bb = ta.volatility.BollingerBands(d["Close"], window=20, window_dev=2)
d["bb_pctb"] = bb.bollinger_pband()

# Precio relativo a media móvil de 20
d["sma20_rel"] = d["Close"] / d["Close"].rolling(20).mean() - 1

# Volatilidad (desvío móvil de retornos, 10 días)
d["vol10"] = d["log_ret"].rolling(10).std()

FEATURES = ["log_ret", "rsi", "macd_diff", "bb_pctb", "sma20_rel", "vol10"]

d = d.dropna().reset_index(drop=False)
print(f"Filas tras calcular indicadores: {len(d)}")
d[FEATURES].describe().round(3)

## 6. Construcción de la etiqueta (SIN data leakage)

Este es el paso metodológico más delicado. Para cada día *t*:

1. Miramos el retorno a **HORIZONTE** días vista: $r_t = \dfrac{\text{Close}_{t+k}}{\text{Close}_t} - 1$
2. Etiquetamos con el umbral $\theta$:
   - $r_t > +\theta \Rightarrow$ **SUBE** (clase 2)
   - $r_t < -\theta \Rightarrow$ **BAJA** (clase 0)
   - en otro caso $\Rightarrow$ **LATERAL** (clase 1)

La etiqueta usa precios **futuros** — eso es correcto porque es lo que queremos predecir.
El *leakage* se evita más adelante, en el **split temporal**: nunca dejamos que días del
período de test aparezcan en el entrenamiento.


In [ ]:
# Retorno futuro a HORIZONTE dias
d["fut_ret"] = d["Close"].shift(-HORIZONTE) / d["Close"] - 1

def etiquetar(r):
    if r > UMBRAL:   return 2   # SUBE
    if r < -UMBRAL:  return 0   # BAJA
    return 1                    # LATERAL

d["label"] = d["fut_ret"].apply(etiquetar)

# Las ultimas HORIZONTE filas no tienen futuro -> se descartan
d = d.dropna(subset=["fut_ret"]).reset_index(drop=True)

# Distribucion de clases
dist = d["label"].value_counts().sort_index()
print("Distribución de clases:")
for k, v in dist.items():
    print(f"  {CLASES[k]:8s}: {v:5d}  ({v/len(d)*100:.1f}%)")

In [ ]:
# Visualizamos el balance de clases
plt.figure(figsize=(6, 4))
plt.bar([CLASES[i] for i in dist.index], dist.values,
        color=["#c0392b", "#7f8c8d", "#27ae60"])
plt.title("Distribución de clases")
plt.ylabel("Cantidad de muestras")
for i, v in enumerate(dist.values):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.show()

## 7. Armado de ventanas y split temporal

Cada muestra de entrada es una **ventana de `VENTANA` días** de los 6 indicadores, aplanada
en un vector de dimensión `VENTANA × 6`. La etiqueta es la dirección del día final de la ventana.

**Split temporal (crítico):** el primer 80% cronológico va a entrenamiento y el último 20% a
prueba. Un `train_test_split` aleatorio mezclaría días futuros en el entrenamiento, produciendo
un *accuracy* artificialmente alto que no se sostiene en producción. Esto es *data leakage*
temporal, el error más frecuente y grave en ML financiero.


In [ ]:
X_raw = d[FEATURES].values
y_all = d["label"].values

X_windows, y_windows = [], []
for i in range(VENTANA, len(d)):
    X_windows.append(X_raw[i - VENTANA:i])   # ventana [i-VENTANA, i)
    y_windows.append(y_all[i - 1])           # etiqueta del ultimo dia de la ventana

X_windows = np.array(X_windows)              # (n, VENTANA, 6)
y_windows = np.array(y_windows)

# Aplanamos la ventana a un vector (modelos densos)
X_flat = X_windows.reshape(X_windows.shape[0], -1)   # (n, VENTANA*6)

# --- Split TEMPORAL con PURGA (embargo) ---
n_test = int(len(X_flat) * TEST_FRAC)
n_train = len(X_flat) - n_test

# Purga: descartamos las ultimas HORIZONTE muestras del train, porque sus
# etiquetas "miran" hasta HORIZONTE dias hacia adelante y se solaparian con
# el inicio del periodo de test (leakage de horizonte en la frontera).
PURGA = HORIZONTE
X_train, X_test = X_flat[:n_train - PURGA], X_flat[n_train:]
y_train, y_test = y_windows[:n_train - PURGA], y_windows[n_train:]

print(f"Train: {X_train.shape} (purga de {PURGA} muestras en la frontera)")
print(f"Test:  {X_test.shape}")
print(f"Dimensión de entrada por muestra: {X_train.shape[1]} features")

### Nota sobre la purga (*purged split*)

Hay una fuga de información sutil que un split temporal simple no elimina: la etiqueta del
último día de entrenamiento se calcula mirando `HORIZONTE` días hacia adelante — días que ya
pertenecen al período de test. Es decir, las últimas etiquetas de train **contienen información
del futuro del test**.

La solución estándar en ML financiero es el ***purged split***: descartar las últimas
`HORIZONTE` muestras del conjunto de entrenamiento, creando una zona de embargo entre train y
test (López de Prado, *Advances in Financial Machine Learning*, 2018, Cap. 7). El costo es
mínimo (perdemos 5 muestras) y elimina el solapamiento por completo.


In [ ]:
# Estandarizacion: ajustamos SOLO con train (evita leakage de estadisticas)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

### Nota sobre estandarización y *leakage*

El `StandardScaler` se ajusta (`fit`) **solo con los datos de entrenamiento**. Si lo
ajustáramos con todo el dataset, las medias y desvíos incorporarían información del período de
test — otra forma sutil de *leakage*. Este cuidado corresponde a la discusión de Goodfellow
sobre la separación estricta entre conjuntos (Cap. 5.3).


## 8. Modelo 1 — Regresión Logística (baseline lineal)

La regresión logística multinomial (*softmax regression*) es el clasificador lineal de
referencia (Goodfellow, Cap. 5.7.1). Modela:

$$P(y = c \mid \mathbf{x}) = \text{softmax}(\mathbf{W}\mathbf{x} + \mathbf{b})_c$$

Su frontera de decisión es un hiperplano. Nos da el **piso de desempeño**: cualquier modelo
más complejo debe superarlo para justificar su costo.


In [ ]:
# Nota: no pasamos multi_class="multinomial" porque el parametro fue eliminado
# en scikit-learn >= 1.7; con el solver por defecto (lbfgs) la regresion
# ya es multinomial (softmax) de forma nativa.
logreg = LogisticRegression(
    max_iter=2000,
    C=1.0,                 # 1/lambda: inverso de la fuerza de regularizacion L2
    random_state=SEED,
)
logreg.fit(X_train_s, y_train)

pred_lr = logreg.predict(X_test_s)
acc_lr = accuracy_score(y_test, pred_lr)
f1_lr  = f1_score(y_test, pred_lr, average="macro")

print(f"Regresión Logística — Accuracy: {acc_lr:.3f} | F1 macro: {f1_lr:.3f}\n")
print(classification_report(y_test, pred_lr,
                            target_names=[CLASES[i] for i in range(3)]))

### Baseline trivial de referencia

Antes de celebrar cualquier *accuracy*, hay que compararlo contra el **clasificador tonto**
que siempre predice la clase mayoritaria. Si el modelo no supera esto, no aprendió nada útil.


In [ ]:
clase_mayoritaria = np.bincount(y_train).argmax()
acc_trivial = (y_test == clase_mayoritaria).mean()
print(f"Clase mayoritaria: {CLASES[clase_mayoritaria]}")
print(f"Accuracy del baseline trivial (siempre '{CLASES[clase_mayoritaria]}'): {acc_trivial:.3f}")

## 9. Modelo 2 — Perceptrón Multicapa (MLP)

El MLP introduce **capas ocultas con activación no lineal** (ReLU), lo que le permite aprender
fronteras de decisión no lineales (Goodfellow, Cap. 6). Aplicamos además técnicas de los
Capítulos 7 y 8:

- **Regularización (Cap. 7):** `Dropout` y penalización L2 sobre los pesos, para combatir el
  sobreajuste — esperable dado el ruido de los datos financieros.
- **Optimización (Cap. 8):** optimizador **Adam**, que adapta la tasa de aprendizaje por
  parámetro, y **early stopping**, que detiene el entrenamiento cuando la pérdida de validación
  deja de mejorar.


In [ ]:
def construir_mlp(input_dim, n_clases=3):
    modelo = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu",
                     kernel_regularizer=keras.regularizers.l2(1e-4)),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu",
                     kernel_regularizer=keras.regularizers.l2(1e-4)),
        layers.Dropout(0.3),
        layers.Dense(n_clases, activation="softmax"),
    ])
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",     # entropia cruzada (Cap. 3)
        metrics=["accuracy"],
    )
    return modelo

mlp = construir_mlp(X_train_s.shape[1])
mlp.summary()

In [ ]:
# Pesos de clase para compensar el desbalance
from sklearn.utils.class_weight import compute_class_weight
pesos = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = {i: w for i, w in enumerate(pesos)}

early = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True)

hist = mlp.fit(
    X_train_s, y_train,
    validation_split=0.2,          # ultimo 20% del train como validacion
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=[early],
    verbose=0,
)
print(f"Entrenamiento detenido en época {len(hist.history['loss'])}")

### 9.1 Curvas de entrenamiento — diagnóstico de sobreajuste (Cap. 7)

Las curvas de pérdida de entrenamiento vs. validación son la herramienta central para
diagnosticar **sobreajuste**. Si la pérdida de validación sube mientras la de entrenamiento
baja, el modelo está memorizando ruido. El *early stopping* corta justo en el mínimo de
validación.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].plot(hist.history["loss"], label="train")
ax[0].plot(hist.history["val_loss"], label="validación")
ax[0].set_title("Pérdida (cross-entropy)")
ax[0].set_xlabel("Época"); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(hist.history["accuracy"], label="train")
ax[1].plot(hist.history["val_accuracy"], label="validación")
ax[1].set_title("Accuracy")
ax[1].set_xlabel("Época"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.show()

In [ ]:
pred_mlp = mlp.predict(X_test_s, verbose=0).argmax(axis=1)
acc_mlp = accuracy_score(y_test, pred_mlp)
f1_mlp  = f1_score(y_test, pred_mlp, average="macro")

print(f"MLP — Accuracy: {acc_mlp:.3f} | F1 macro: {f1_mlp:.3f}\n")
print(classification_report(y_test, pred_mlp,
                            target_names=[CLASES[i] for i in range(3)]))

## 10. Comparativa de modelos

Reunimos los resultados de ambos modelos y del baseline trivial en una tabla y en gráficos.


In [ ]:
resultados = pd.DataFrame({
    "Modelo":   ["Baseline trivial", "Regresión Logística", "MLP"],
    "Accuracy": [acc_trivial, acc_lr, acc_mlp],
    "F1 macro": [np.nan, f1_lr, f1_mlp],
}).round(3)
resultados

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

modelos = resultados["Modelo"]
ax[0].bar(modelos, resultados["Accuracy"], color=["#95a5a6", "#2980b9", "#8e44ad"])
ax[0].axhline(acc_trivial, ls="--", color="red", alpha=0.6, label="baseline trivial")
ax[0].set_title("Accuracy por modelo"); ax[0].set_ylim(0, max(0.6, resultados["Accuracy"].max()*1.2))
ax[0].legend()
for i, v in enumerate(resultados["Accuracy"]):
    ax[0].text(i, v, f"{v:.3f}", ha="center", va="bottom")

# Matriz de confusion del MLP
cm = confusion_matrix(y_test, pred_mlp)
im = ax[1].imshow(cm, cmap="Blues")
ax[1].set_title("Matriz de confusión — MLP")
ax[1].set_xticks(range(3)); ax[1].set_xticklabels([CLASES[i] for i in range(3)])
ax[1].set_yticks(range(3)); ax[1].set_yticklabels([CLASES[i] for i in range(3)])
ax[1].set_xlabel("Predicho"); ax[1].set_ylabel("Real")
for i in range(3):
    for j in range(3):
        ax[1].text(j, i, cm[i, j], ha="center",
                   color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout(); plt.show()

## 11. Demostración: predicción sobre un gráfico nuevo

Cerramos con lo pedido en la consigna: tomamos una ventana **del período de test** (que ningún
modelo vio durante el entrenamiento), la mostramos como gráfico de precios, y comparamos qué
predice cada modelo contra lo que realmente ocurrió.

> Nota: acá el gráfico de velas es una **visualización del resultado**, no la entrada del modelo.
> Los modelos consumen los indicadores numéricos, no la imagen — esto es lo metodológicamente correcto.


In [ ]:
# Elegimos una muestra del test para inspeccionar
idx_local = len(X_test_s) // 2          # muestra intermedia del test

# Mapeo de indice: la muestra j de X_windows tiene su etiqueta en el dia
# global VENTANA + j - 1 (ultimo dia de la ventana). Para el test, j = n_train + idx_local.
idx_global = VENTANA + (n_train + idx_local) - 1

# Verificacion de alineacion (si falla, hay un error de indexado)
assert y_windows[n_train + idx_local] == y_test[idx_local]
assert d.loc[idx_global, "label"] == y_test[idx_local], "desalineacion de indices!"

x_muestra = X_test_s[idx_local:idx_local+1]
real = y_test[idx_local]

p_lr  = logreg.predict(x_muestra)[0]
p_mlp = mlp.predict(x_muestra, verbose=0).argmax(axis=1)[0]
probs_mlp = mlp.predict(x_muestra, verbose=0)[0]

print(f"Fecha del día evaluado: {d.loc[idx_global, 'Date'].date()}")
print(f"Real:                 {CLASES[real]}")
print(f"Regresión Logística:  {CLASES[p_lr]}")
print(f"MLP:                  {CLASES[p_mlp]}  (probs: "
      f"BAJA={probs_mlp[0]:.2f}, LATERAL={probs_mlp[1]:.2f}, SUBE={probs_mlp[2]:.2f})")

In [ ]:
# Graficamos la ventana + el horizonte futuro que se intentaba predecir
ini = idx_global - VENTANA
fin = min(idx_global + HORIZONTE, len(d) - 1)   # clamp: evita salirnos del dataframe
tramo = d.iloc[ini:fin+1]

plt.figure(figsize=(13, 5))
plt.plot(tramo["Date"], tramo["Close"], marker="o", ms=3, linewidth=1)
plt.axvline(d.loc[idx_global, "Date"], color="black", ls="--",
            label="día de la predicción")
plt.axvspan(d.loc[idx_global, "Date"], d.loc[fin, "Date"],
            alpha=0.12, color="orange", label=f"horizonte futuro ({HORIZONTE}d)")
plt.title(f"{TICKER} — Ventana evaluada  |  Real: {CLASES[real]}  •  "
          f"LogReg: {CLASES[p_lr]}  •  MLP: {CLASES[p_mlp]}")
plt.ylabel("Precio"); plt.legend(); plt.grid(alpha=0.3)
plt.xticks(rotation=45); plt.tight_layout()
plt.show()

## 12. Conclusiones

**Sobre la comparativa de modelos.** El MLP suele igualar o superar levemente a la regresión
logística en *F1 macro*, lo que sugiere que existe algo de estructura no lineal aprovechable en
los indicadores técnicos. Sin embargo, la mejora es modesta: es esperable, porque la predicción
de dirección de precios está muy cerca del límite de lo predecible en un mercado razonablemente
eficiente. Este resultado, lejos de ser un fracaso, **es en sí mismo un hallazgo** y conecta con
la discusión de Goodfellow sobre la relación entre **capacidad del modelo y estructura real del
problema** (Cap. 5.2): más capacidad sólo ayuda si hay señal que capturar.

**Sobre la metodología.** Los puntos que blindan este trabajo metodológicamente:
- Split **temporal** (no aleatorio), evitando *data leakage* de futuro.
- **Purga (embargo)** de `HORIZONTE` muestras en la frontera train/test, eliminando el
  solapamiento entre las últimas etiquetas de entrenamiento y el inicio del test.
- `StandardScaler` ajustado **solo con train**.
- Validación de Keras (`validation_split`) que toma el **último tramo** del train sin
  mezclar, preservando el orden temporal también en la validación interna.
- Comparación contra un **baseline trivial** (clase mayoritaria), no contra el 0%.
- Uso de **F1 macro** además de *accuracy*, robusto frente al desbalance de clases.

**Conceptos de Goodfellow demostrados.** Tarea/desempeño/experiencia (Cap. 5); frontera lineal
vs. no lineal y aproximación universal (Cap. 6); regularización L2, dropout y early stopping
(Cap. 7); optimización con Adam y descenso de gradiente estocástico (Cap. 8); entropía cruzada
como máxima verosimilitud y softmax como distribución de probabilidad (Cap. 3).

**Extensiones posibles.** Añadir una CNN 1D (Cap. 9) para explotar la estructura local de la
serie, o una LSTM (Cap. 10) para modelar dependencias temporales explícitas, ampliaría la
comparativa hacia arquitecturas de *deep learning* propiamente dichas.

---

### Referencia

Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press.
